<a href="https://colab.research.google.com/github/sylle321/Hotel_booking/blob/main/ba305_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px                          # interactive plots (pre-installed on Colab)
from sklearn.preprocessing import StandardScaler     # does the scaling for us
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm
from sklearn.pipeline import Pipeline
# Libraries used in the Session 13 decision tree lab
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
path = "/content/hotels.csv"
df = pd.read_csv(path)
#Shows each columns data type and number of non-null values.
df.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/hotels.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Counts distinct values in each column of the dataframe.
df.nunique()

In [ ]:
# Convert hotel to a dummy variable:
# Resort Hotel = 0, City Hotel = 1
df['hotel'] = df['hotel'].map({
    'Resort Hotel': 0,
    'City Hotel': 1
})

df.head()

# Create dummy variables for the other categorical variables
categorical_cols = [
    'deposit_type',
    'market_segment',
    'customer_type',
    'reserved_room_type'
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

df_encoded.info()

In [ ]:
# Selecting the columns of interest for analysis
# Making sure unrelated columns are not included in the analysis.
cols = [
    "hotel",
    "is_canceled",
    "lead_time",
    "deposit_type",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "customer_type",
    "booking_changes",
    "is_repeated_guest",
    "market_segment",
    "total_of_special_requests",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "reserved_room_type",
    "adr",
    "distribution_channel",
    "arrival_date_month"
]

hotel_selected = df[cols]

hotel_selected.info()

In [ ]:
#Cleaning the data by removing rows with any missing values and resetting the index.
hotel_clean = hotel_selected.dropna(how="any") #Removes rows with any missing values.
hotel_clean = hotel_clean.reset_index(drop=True) #resets the index

print(hotel_selected.shape)
print(hotel_clean.shape)

In [ ]:
#Checking cancelation counts and proportions in the cleaned dataset.
print(hotel_clean["is_canceled"].value_counts()) #Gives the count of each unique value in the "is_canceled" column.

print(
    hotel_clean["is_canceled"].value_counts(normalize=True) # Gives the proportion of each unique value in the "is_canceled" column.
)


In [ ]:
# Calculate the correlation using PCA lab's code as reference.
# We exclude categorical string columns and only keep truly numeric columns
numeric_cols = [
    "is_canceled",
    "lead_time",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "is_repeated_guest",
    "total_of_special_requests",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr"
]

hotel_numeric = hotel_clean[numeric_cols]

corr_mat = hotel_numeric.corr()

corr_mat.round(4)

In [ ]:
# Correlation with is_canceled only
corr_mat[["is_canceled"]].round(3)

In [ ]:
# Separate the target variable from the predictor variables
knn_cols = hotel_numeric

X = knn_cols.drop(columns=["is_canceled"])
y = knn_cols["is_canceled"]

# Split the data into 80% training and 20% testing
# stratify maintains the same cancellation proportion in both sets
# random_state ensures the split is reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

# Check the size of each dataset
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [ ]:
# Combine scaling and KNN in one pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

# Compare different numbers of neighbors
k_values = list(range(1, 31))

# Choose k using cross-validation on training data
knn_search = GridSearchCV(
    knn_pipeline,
    {"knn__n_neighbors": k_values},
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Learn scaling separately within each training fold
knn_search.fit(X_train, y_train)

# Retrieve the best model, refitted on all training data
knn = knn_search.best_estimator_

print("Best k:", knn_search.best_params_["knn__n_neighbors"])
print(
    "Best cross-validation accuracy:",
    round(knn_search.best_score_, 3)
)

# Extract average validation accuracy for each k
accuracies = knn_search.cv_results_["mean_test_score"]

# Plot cross-validation accuracy
plt.figure(figsize=(10, 6))
plt.scatter(k_values, accuracies)
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Cross-validation accuracy")
plt.title("KNN Accuracy vs. Number of Neighbors")
plt.xticks(k_values)
plt.grid(True)
plt.show()

# Predict test bookings; the pipeline handles scaling
y_pred = knn.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

In [ ]:
baseline_accuracy = y_test.value_counts(normalize=True).max()
print("Baseline Accuracy:", baseline_accuracy)
# calculate the baseline accuracy
# compare with the knn-model accuracy
# knn-model's accuracy greater than the baseline


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [15]:
# Create a full tree without limits on depth or leaf size
full_clf = DecisionTreeClassifier(random_state=0)

# Learn cancellation patterns from the training data
full_clf.fit(X_train, y_train)

# Count all decision nodes and leaves
print("Number of nodes:", full_clf.tree_.node_count)

# Predict outcomes for training and test bookings
y_pred_train = full_clf.predict(X_train)
y_pred_full = full_clf.predict(X_test)

# Measure how closely the tree fits the training data
print(
    "Training accuracy:",
    round(accuracy_score(y_train, y_pred_train), 3)
)

# Measure performance on bookings not used for training
print(
    "Testing accuracy:",
    round(accuracy_score(y_test, y_pred_full), 3)
)

# Rows = actual classes; columns = predicted classes
# Class order: 0 = not canceled, 1 = canceled
print("Confusion matrix on test data:")
print(confusion_matrix(y_test, y_pred_full, labels=[0, 1]))

NameError: name 'X_train' is not defined

In [ ]:
# Set the size of the tree diagram
plt.figure(figsize=(22, 10))

tree.plot_tree(
    full_clf,
    max_depth=2,  # Display the top three levels; the model stays unchanged
    feature_names=list(X_train.columns),  # Label splits with predictor names
    class_names=["Not canceled", "Canceled"],  # Label the two outcomes
    filled=True,  # Color nodes by their class composition
    fontsize=9
)

# Display the diagram
plt.show()

In [ ]:
# Create a tree using the settings selected by cross-validation
clf = DecisionTreeClassifier(
    max_depth=20,              # Allow deeper patterns than the previous depth of 10
    min_samples_split=40,      # A node needs at least 40 bookings before splitting
    min_samples_leaf=20,       # Each leaf must contain at least 20 bookings
    min_impurity_decrease=0.01,   # Allow small improvements to justify a split
    criterion="gini",          # Prefer splits that separate the two classes
    random_state=0             # Make the results reproducible
)

# Learn cancellation patterns from the training data
clf.fit(X_train, y_train)

# Predict outcomes for training and test bookings
y_pred_train_tree = clf.predict(X_train)
y_pred_tree = clf.predict(X_test)

# Compare training and test accuracy to assess generalization
print(
    "Training accuracy:",
    round(accuracy_score(y_train, y_pred_train_tree), 3)
)

print(
    "Testing accuracy:",
    round(accuracy_score(y_test, y_pred_tree), 3)
)

# Rows = actual outcomes; columns = predicted outcomes
# Class order: 0 = not canceled, 1 = canceled
print("Confusion matrix on test data:")
print(confusion_matrix(y_test, y_pred_tree, labels=[0, 1]))

In [ ]:
# Create a tree using the settings selected by cross-validation
clf = DecisionTreeClassifier(
    max_depth=20,              # Allow deeper patterns than the previous depth of 10
    min_samples_split=40,      # A node needs at least 40 bookings before splitting
    min_samples_leaf=20,       # Each leaf must contain at least 20 bookings
    min_impurity_decrease=0,   # Allow smaller improvements to justify a split
    criterion="gini",          # Prefer splits that separate the two classes
    random_state=0             # Make the results reproducible
)

# Learn cancellation patterns from the training data
clf.fit(X_train, y_train)

# Predict outcomes for training and test bookings
y_pred_train_tree = clf.predict(X_train)
y_pred_tree = clf.predict(X_test)

# Compare training and test accuracy to assess generalization
print(
    "Training accuracy:",
    round(accuracy_score(y_train, y_pred_train_tree), 3)
)

print(
    "Testing accuracy:",
    round(accuracy_score(y_test, y_pred_tree), 3)
)

# Rows = actual outcomes; columns = predicted outcomes
# Class order: 0 = not canceled, 1 = canceled
print("Confusion matrix on test data:")
print(confusion_matrix(y_test, y_pred_tree, labels=[0, 1]))

In [ ]:
# Train and validate the tree across five folds of the training data
accuracy_scores = cross_val_score(
    clf,
    X_train,
    y_train,
    cv=5,                # Use five validation rounds
    scoring="accuracy"   # Evaluate the proportion of correct predictions
)

# Display accuracy for each validation fold
print(
    "Accuracy scores of each fold:",
    [round(acc, 3) for acc in accuracy_scores]
)

# Calculate average accuracy across the five folds
print(
    "5-fold cross-validation mean accuracy:",
    round(accuracy_scores.mean(), 3)
)

In [ ]:
# Define the parameter values to compare
param_grid = {
    "max_depth": [5, 10, 20],               # Compare different depth limits
    "min_samples_leaf": [20, 50, 100],      # Compare minimum leaf sizes
    "min_impurity_decrease": [0, 0.001, 0.01]  # Compare split restrictions
}

# Compare all 27 combinations using five-fold cross-validation
gridSearch = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid,
    cv=5,                # Validate each combination across five folds
    scoring="accuracy",  # Select settings with the highest mean accuracy
    n_jobs=-1            # Use all available CPU cores
)

# Search using only the training data
gridSearch.fit(X_train, y_train)

# Display the best validation score and parameter combination
print("Best cross-validation accuracy:", gridSearch.best_score_)
print("Best parameters:", gridSearch.best_params_)

# Retrieve the best model, automatically refitted on all training data
best_clf = gridSearch.best_estimator_

# Predict outcomes for the held-out test bookings
y_pred_best_tree = best_clf.predict(X_test)

In [ ]:
# Create a forest that combines predictions from 500 trees
rf = RandomForestClassifier(
    n_estimators=500,  # Number of trees in the forest
    random_state=0,   # Make the results reproducible
    criterion="gini", # Evaluate splits using Gini impurity
    n_jobs=-1         # Train trees using all available CPU cores
)

# Train the forest using the training bookings
rf.fit(X_train, y_train)

# Predict cancellation outcomes for the test bookings
y_pred_rf = rf.predict(X_test)

# Measure how closely the forest fits the training data
print(
    "Training accuracy:",
    round(accuracy_score(y_train, rf.predict(X_train)), 3)
)

# Measure performance on unseen test bookings
print(
    "Testing accuracy:",
    round(accuracy_score(y_test, y_pred_rf), 3)
)

# Show correct predictions, false alarms, and missed cancellations
print("Confusion matrix on test data:")
print(confusion_matrix(y_test, y_pred_rf, labels=[0, 1]))

In [ ]:
# Find the most common outcome in the training data
majority_class = y_train.mode().iloc[0]

# Baseline: predict this outcome for every test booking
y_pred_baseline = np.full(len(y_test), majority_class)

# Collect each model's predictions for the same test bookings
predictions = {
    "Baseline": y_pred_baseline,
    "Full decision tree": y_pred_full,
    "Reduced decision tree": y_pred_tree,
    "Tuned decision tree": y_pred_best_tree,
    "Random forest": y_pred_rf
}

# Create an empty list to store evaluation results
results = []

# Calculate the same metrics for each model
for model_name, prediction in predictions.items():
    results.append({
        "Model": model_name,

        # Proportion of all bookings classified correctly
        "Accuracy": accuracy_score(y_test, prediction),

        # Proportion of flagged cancellations that actually canceled
        "Precision": precision_score(
            y_test, prediction, zero_division=0
        ),

        # Proportion of actual cancellations identified by the model
        "Recall": recall_score(
            y_test, prediction, zero_division=0
        ),

        # Harmonic mean of cancellation precision and recall
        "F1": f1_score(
            y_test, prediction, zero_division=0
        )
    })

# Convert the results into a table with model names as row labels
results_df = pd.DataFrame(results).set_index("Model")

# Display metrics rounded to three decimal places
results_df.round(3)

In [ ]:
# Match each predictor with its impurity-based importance
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

# Rank predictors from highest to lowest importance
importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

# Display the importance values
print(importance_df)

# Plot the ranked predictor importance
importance_df.plot(
    kind="bar",
    x="feature",
    y="importance",
    figsize=(10, 5),
    legend=False
)

# Add labels and adjust spacing
plt.ylabel("Feature importance")
plt.title("Random Forest: Importance of Cancellation Predictors")
plt.tight_layout()
plt.show()

# Importance describes split contributions, not direction or causation

In [ ]:
# Evaluate the tree selected through cross-validation
print(
    "Tuned tree test accuracy:",
    round(accuracy_score(y_test, y_pred_best_tree), 3)
)

# Show correct and incorrect predictions for each class
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_best_tree, labels=[0, 1]))